In [ ]:
# Step 1: Setup and Data Loading

import pandas as pd
import numpy as np

# File paths (update with your actual Google Drive paths or upload to Colab)
files = [
    "/content/yellow_tripdata_2025-04.parquet",
    "/content/yellow_tripdata_2025-05.parquet",
    "/content/yellow_tripdata_2025-06.parquet",
    "/content/yellow_tripdata_2025-07.parquet"
]

# Load all months into a single DataFrame
df_list = [pd.read_parquet(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

print("Shape of combined data:", df.shape)
df.head()
df.info()


Shape of combined data: (16784321, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16784321 entries, 0 to 16784320
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   f

In [ ]:
# Step 2 : Datetime Parsing & Robust Cleaning

# Convert pickup & dropoff times to datetime
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')

# Drop rows with missing or invalid datetimes
df = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

# Remove trips with invalid durations (< 2 min or > 12 hrs)
df = df[(df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() >= 120]
df = df[(df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() <= 12*3600]

# Remove trips with non-positive or extreme distances
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] <= 100)]

# Remove trips with non-positive fare amount
df = df[df['fare_amount'] > 0]

# Remove trips with non-positive total amount
df = df[df['total_amount'] > 0]

# Keep only realistic passenger counts (1–6)
df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 6)]

# Keep only valid TLC zone IDs (1–263 for 2025 dataset)
df = df[(df['PULocationID'].between(1, 263)) & (df['DOLocationID'].between(1, 263))]

print("Shape after revised cleaning:", df.shape)
df.head()


In [ ]:
# Step 3 : Time-Based Feature Engineering with cyclical encoding

# Basic time features
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.weekday  # 0=Mon, 6=Sun
df['is_weekend'] = df['pickup_weekday'].isin([5, 6]).astype(int)

# Cyclical encoding for hour (24-hour cycle)
df['pickup_hour_sin'] = np.sin(2 * np.pi * df['pickup_hour'] / 24)
df['pickup_hour_cos'] = np.cos(2 * np.pi * df['pickup_hour'] / 24)

# Cyclical encoding for weekday (7-day cycle)
df['pickup_weekday_sin'] = np.sin(2 * np.pi * df['pickup_weekday'] / 7)
df['pickup_weekday_cos'] = np.cos(2 * np.pi * df['pickup_weekday'] / 7)

print("Shape after adding time features:", df.shape)
df[['tpep_pickup_datetime', 'pickup_hour', 'pickup_weekday', 'is_weekend',
    'pickup_hour_sin', 'pickup_hour_cos',
    'pickup_weekday_sin', 'pickup_weekday_cos']].head()


In [ ]:
# Step 4: Aggregate demand per zone per hour

# Round pickup time to the hour
df['pickup_hourly'] = df['tpep_pickup_datetime'].dt.floor('H')

# Group by hour + pickup location
demand = (
    df.groupby(['pickup_hourly', 'PULocationID'])
      .size()
      .reset_index(name='demand_count')
)

print("Shape of aggregated demand:", demand.shape)
demand.head(10)


In [ ]:
# Step 5: Add time-based features to aggregated demand

# Extract features from pickup_hourly
demand['hour'] = demand['pickup_hourly'].dt.hour
demand['day'] = demand['pickup_hourly'].dt.day
demand['weekday'] = demand['pickup_hourly'].dt.weekday  # 0=Mon, 6=Sun
demand['is_weekend'] = demand['weekday'].isin([5, 6]).astype(int)

# Cyclical encoding for hour
demand['hour_sin'] = np.sin(2 * np.pi * demand['hour'] / 24)
demand['hour_cos'] = np.cos(2 * np.pi * demand['hour'] / 24)

# Cyclical encoding for weekday
demand['weekday_sin'] = np.sin(2 * np.pi * demand['weekday'] / 7)
demand['weekday_cos'] = np.cos(2 * np.pi * demand['weekday'] / 7)

print("Shape after joining time features:", demand.shape)
demand.head(10)


In [ ]:
# Step 6: Add optional aggregated features (distance, fare, passengers)

agg_features = (
    df.groupby(['pickup_hourly', 'PULocationID'])
      .agg(
          avg_trip_distance=('trip_distance', 'mean'),
          avg_fare_amount=('fare_amount', 'mean'),
          avg_passenger_count=('passenger_count', 'mean')
      )
      .reset_index()
)

# Merge with the demand DataFrame
demand = demand.merge(agg_features, on=['pickup_hourly', 'PULocationID'], how='left')

print("Shape after adding aggregated features:", demand.shape)
demand.head(10)


In [ ]:
# Step 7: Exploratory Data Analysis (EDA)

import matplotlib.pyplot as plt

# 1. Demand by Hour of Day
hourly_demand = demand.groupby('hour')['demand_count'].mean()
plt.figure(figsize=(8,4))
hourly_demand.plot(kind='bar')
plt.title("Average Taxi Demand by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Average Demand")
plt.show()

# 2. Demand by Day of Week
weekday_demand = demand.groupby('weekday')['demand_count'].mean()
plt.figure(figsize=(8,4))
weekday_demand.plot(kind='bar')
plt.title("Average Taxi Demand by Day of Week (0=Mon, 6=Sun)")
plt.xlabel("Day of Week")
plt.ylabel("Average Demand")
plt.show()

# 3. Top 10 Zones by Demand
zone_demand = demand.groupby('PULocationID')['demand_count'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,5))
zone_demand.plot(kind='bar')
plt.title("Top 10 Pickup Zones by Total Demand")
plt.xlabel("Zone ID")
plt.ylabel("Total Demand")
plt.show()

# 4. Feature Ranges for Optional Features (if present)
if {'avg_trip_distance','avg_fare_amount','avg_passenger_count'}.issubset(demand.columns):
    print("Feature Ranges:")
    print("Avg Trip Distance:", demand['avg_trip_distance'].min(), "to", demand['avg_trip_distance'].max())
    print("Avg Fare Amount:", demand['avg_fare_amount'].min(), "to", demand['avg_fare_amount'].max())
    print("Avg Passenger Count:", demand['avg_passenger_count'].min(), "to", demand['avg_passenger_count'].max())


In [ ]:
# Step 8: Scaling + Encoding

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define target
y = demand['demand_count']

# Feature sets
numeric_features = ['avg_trip_distance', 'avg_fare_amount', 'avg_passenger_count']
cyclical_features = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos']
basic_features = ['hour', 'weekday', 'is_weekend']
categorical_features = ['PULocationID']

# Select columns that exist (in case optional features were skipped)
numeric_features = [col for col in numeric_features if col in demand.columns]

# Final feature set
feature_cols = numeric_features + cyclical_features + basic_features + categorical_features

X = demand[feature_cols]

# --- Pipeline for Linear Regression (scaled) ---
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('pass', 'passthrough', cyclical_features + basic_features)
    ]
)

X_scaled_lr = preprocessor_lr.fit_transform(X)

# --- Pipeline for XGBoost (no scaling) ---
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('pass', 'passthrough', cyclical_features + basic_features)
    ]
)

X_xgb = preprocessor_xgb.fit_transform(X)

print("Shapes -> LR:", X_scaled_lr.shape, " | XGBoost:", X_xgb.shape)


In [ ]:
# Step 9: Final Train/Test Split + Save Artifacts

import joblib
from sklearn.model_selection import train_test_split

# ----------------------------
# 1. Time-based Train/Test Split
# ----------------------------

# Sort by time
demand = demand.sort_values("pickup_hourly")

# Define cutoff (e.g., last 2 weeks as test set)
cutoff_date = demand['pickup_hourly'].max() - pd.Timedelta(days=14)
train_idx = demand['pickup_hourly'] < cutoff_date
test_idx = demand['pickup_hourly'] >= cutoff_date

# Train/test splits
X_train_lr, X_test_lr = X_scaled_lr[train_idx], X_scaled_lr[test_idx]
X_train_xgb, X_test_xgb = X_xgb[train_idx], X_xgb[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train_lr.shape[0], "| Test size:", X_test_lr.shape[0])

# ----------------------------
# 2. Save cleaned dataset
# ----------------------------
demand.to_parquet("/content/cleaned_demand_dataset.parquet", index=False)
print("Saved cleaned dataset -> cleaned_demand_dataset.parquet")

# ----------------------------
# 3. Save preprocessing pipelines
# ----------------------------
joblib.dump(preprocessor_lr, "/content/preprocessor_lr.pkl")
joblib.dump(preprocessor_xgb, "/content/preprocessor_xgb.pkl")

print("Saved preprocessors -> preprocessor_lr.pkl, preprocessor_xgb.pkl")

# ----------------------------
# 4. Save train/test sets (optional, if team prefers direct arrays)
# ----------------------------
joblib.dump((X_train_lr, X_test_lr, y_train, y_test), "/content/linear_reg_data.pkl")
joblib.dump((X_train_xgb, X_test_xgb, y_train, y_test), "/content/xgb_data.pkl")

print("Saved train/test sets -> linear_reg_data.pkl, xgb_data.pkl")
